# 02 - Data preprocessing
Order matters: **split first**, then fit the scaler on the training rows only, then oversample the training rows only. The old notebook scaled before splitting (leakage) and saved the *un-resampled* data under `*_smote.csv` names.

In [ ]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("../data/processed/clean_creditcard.csv")   # written by 01_EDA.ipynb
X = df.drop(columns="Class")
y = df["Class"]
print(X.shape, y.value_counts().to_dict())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("train:", X_train.shape, " test:", X_test.shape)

In [ ]:
# Fit the scaler on TRAIN only, then apply it to both splits
scaler = StandardScaler().fit(X_train[["Amount"]])
X_train = X_train.copy(); X_test = X_test.copy()
X_train["Amount"] = scaler.transform(X_train[["Amount"]]).ravel()
X_test["Amount"] = scaler.transform(X_test[["Amount"]]).ravel()

In [ ]:
# Oversample the TRAINING split only - never the test set
X_train_smote, y_train_smote = SMOTE(random_state=42).fit_resample(X_train, y_train)
print("before:", y_train.value_counts().to_dict())
print("after :", y_train_smote.value_counts().to_dict())

In [ ]:
X_train_smote.to_csv("../data/processed/X_train_smote.csv", index=False)
y_train_smote.to_csv("../data/processed/y_train_smote.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
joblib.dump(scaler, "../models/scaler.pkl")
print("Preprocessing completed.")